In [10]:
# !pip install -q torch_geometric
# !pip install -q class_resolver
# !pip3 install pymatting

In [11]:
# from google.colab import drive
# drive.mount('/content/drive')

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as nnFn
import torch.optim as optim
import numpy as np
import random
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch_geometric.data import Data
from torch_geometric.nn import ARMAConv
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss
)
from torch.utils.data import TensorDataset, DataLoader, Subset
from torch_geometric.data import Data
from torch_geometric.nn import ARMAConv
from torch.utils.data import TensorDataset, DataLoader, Subset
from torchvision import models

In [13]:
import torch
print("CUDA available:", torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU Name: NVIDIA RTX A4000


In [14]:
import numpy as np

feature_path = "/home/snu/Downloads/liver_dino_features.npy"

label_path = "/home/snu/Downloads/liver_dino_labels.npy"

features = np.load(
    feature_path
).astype(np.float32)

y_labels = np.load(
    label_path
).astype(np.int64)

print("Feature shape:", features.shape)

print("Label shape:", y_labels.shape)

print("\nClass distribution:")

unique, counts = np.unique(
    y_labels,
    return_counts=True
)

for cls, cnt in zip(unique, counts):

    print(f"Class {cls}: {cnt}")

Feature shape: (635, 768)
Label shape: (635,)

Class distribution:
Class 0: 200
Class 1: 435


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as nnFn
from torch_geometric.nn import ARMAConv

class ARMA_SemiSupervised(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, device, num_stacks=1, num_layers=3):
        super(ARMA_SemiSupervised, self).__init__()
        self.device = device

        self.conv1 = ARMAConv(input_dim, hidden_dim, num_stacks=num_stacks, num_layers=num_layers,
                              shared_weights=True, dropout=0.3)
        self.conv2 = ARMAConv(hidden_dim, hidden_dim, num_stacks=num_stacks, num_layers=num_layers,
                              shared_weights=True, dropout=0.3)
        self.conv3 = ARMAConv(hidden_dim, hidden_dim, num_stacks=num_stacks, num_layers=num_layers,
                              shared_weights=True, dropout=0.3)

        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.num_clusters = output_dim

        activations = {
            "SELU": nnFn.selu,
            "SiLU": nnFn.silu,
            "GELU": nnFn.gelu,
            "RELU": nnFn.relu,
            "ELU": nnFn.elu,
            "PReLU": nnFn.prelu,
            "LeakyReLU": nnFn.leaky_relu
        }
        self.act = activations.get("RELU", nnFn.relu)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # --- Layer 1 ---
        x = self.conv1(x, edge_index)
        x = self.act(x)

        x = self.dropout(x)

        # --- Layer 2 ---
        x = self.conv2(x, edge_index)
        x = self.act(x)
        x = self.dropout(x)

        # --- Layer 3 ---
        x = self.conv3(x, edge_index)
        x = self.act(x)
        x = self.dropout(x)

        logits = self.fc(x)
        return logits

    def cut_loss(self, A, S):
        S = nnFn.softmax(S, dim=1)
        A_pool = torch.matmul(torch.matmul(A, S).t(), S)
        num = torch.trace(A_pool)

        D = torch.diag(torch.sum(A, dim=-1))
        D_pooled = torch.matmul(torch.matmul(D, S).t(), S)
        den = torch.trace(D_pooled)
        mincut_loss = -(num / den)

        St_S = torch.matmul(S.t(), S)
        I_S = torch.eye(self.num_clusters, device=self.device)
        ortho_loss = torch.norm(St_S / torch.norm(St_S) - I_S / torch.norm(I_S))

        return mincut_loss + ortho_loss


In [16]:
def create_adj(F, alpha=1):
    F_norm = F / np.linalg.norm(F, axis=1, keepdims=True)
    W = np.dot(F_norm, F_norm.T)
    W = np.where(W >= alpha, 1, 0).astype(np.float32)
    W = W / W.max()
    return W

def load_data(adj, node_feats):
    node_feats = torch.from_numpy(node_feats).float()
    edge_index = torch.from_numpy(np.array(np.nonzero((adj > 0))))
    return Data(x=node_feats, edge_index=edge_index)

In [17]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
alpha = 0.83
feats_dim = features.shape[1]
F = features
y = y_labels
hidden_dim = 256
num_classes = 2
num_epochs = 2000
lr = 0.0001
weight_decay = 1e-4
batch_print_freq = 100
lambda_mod =  0.001  #0.2 #0.001  # weight for modularity loss
# lambda_sup = 1.0

In [18]:
W = create_adj(F, alpha)
data = load_data(W, F).to(device)
print(data)

Data(x=[635, 768], edge_index=[2, 65773])


In [19]:
sss = StratifiedShuffleSplit(n_splits=20, test_size=0.9, random_state=42)

accuracies, precisions, recalls, f1_scores, aucs, ce_losses = [], [], [], [], [], []

A_tensor = torch.from_numpy(W).float().to(device)

for fold, (train_idx, test_idx) in enumerate(sss.split(F, y), start=1):
    print(f"\n=== Fold {fold} ===")

    train_idx_t = torch.from_numpy(train_idx).long().to(device)
    y_train_tensor = torch.from_numpy(y[train_idx]).long().to(device)

    model = ARMA_SemiSupervised(feats_dim, hidden_dim, num_classes, device).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce_loss = nn.CrossEntropyLoss()

    # Training
    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(data)
        loss_sup = ce_loss(logits[train_idx_t], y_train_tensor)
        loss_unsup = model.cut_loss(A_tensor, logits)
        total_loss = loss_sup + lambda_mod * loss_unsup

        total_loss.backward()
        optimizer.step()

        if epoch % batch_print_freq == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                preds_train = logits[train_idx_t].argmax(dim=1)
                acc_train = accuracy_score(y_train_tensor.cpu(), preds_train.cpu())
            print(f"Fold {fold} Epoch {epoch}: "
                  f"TotalLoss={total_loss.item():.6f} | Sup={loss_sup.item():.6f} | "
                  f"Unsup={loss_unsup.item():.6f} | TrainAcc={acc_train:.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        out = model(data)
        preds = out.argmax(dim=1).cpu().numpy()
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()  # Probability for class 1

    y_test = y[test_idx]
    y_pred_test = preds[test_idx]
    y_prob_test = probs[test_idx]

    acc = accuracy_score(y_test, y_pred_test)
    prec = precision_score(y_test, y_pred_test, zero_division=0)
    rec = recall_score(y_test, y_pred_test, zero_division=0)
    f1 = f1_score(y_test, y_pred_test, zero_division=0)
    auc = roc_auc_score(y_test, y_prob_test)
    ce = log_loss(y_test, y_prob_test)

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)
    aucs.append(auc)
    ce_losses.append(ce)

    print(f"Fold {fold} → "
          f"Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | "
          f"F1={f1:.4f} | AUC={auc:.4f} | CE Loss={ce:.4f}")


# Final summary
print("\n=== Average Results Across 20 Folds ===")
print(f"Accuracy:  {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"Recall:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"F1-score:  {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC:       {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"CE Loss:   {np.mean(ce_losses):.4f} ± {np.std(ce_losses):.4f}")


=== Fold 1 ===
Fold 1 Epoch 1: TotalLoss=1.436866 | Sup=1.437139 | Unsup=-0.272978 | TrainAcc=0.4444
Fold 1 Epoch 100: TotalLoss=0.505355 | Sup=0.505655 | Unsup=-0.299775 | TrainAcc=0.8095
Fold 1 Epoch 200: TotalLoss=0.190167 | Sup=0.190495 | Unsup=-0.328162 | TrainAcc=0.9206
Fold 1 Epoch 300: TotalLoss=0.194676 | Sup=0.195055 | Unsup=-0.378939 | TrainAcc=0.9206
Fold 1 Epoch 400: TotalLoss=0.091103 | Sup=0.091518 | Unsup=-0.414808 | TrainAcc=0.9524
Fold 1 Epoch 500: TotalLoss=0.092159 | Sup=0.092526 | Unsup=-0.367789 | TrainAcc=0.9365
Fold 1 Epoch 600: TotalLoss=0.081507 | Sup=0.081885 | Unsup=-0.378180 | TrainAcc=0.9683
Fold 1 Epoch 700: TotalLoss=0.066732 | Sup=0.067077 | Unsup=-0.344809 | TrainAcc=0.9524
Fold 1 Epoch 800: TotalLoss=0.009312 | Sup=0.009668 | Unsup=-0.355743 | TrainAcc=1.0000
Fold 1 Epoch 900: TotalLoss=0.009115 | Sup=0.009493 | Unsup=-0.377952 | TrainAcc=1.0000
Fold 1 Epoch 1000: TotalLoss=0.036563 | Sup=0.036899 | Unsup=-0.336035 | TrainAcc=0.9841
Fold 1 Epoch 1100